1. STAGING ORDER VIEWS

In [0]:
drop view if exists stg_order_lines;
CREATE OR REPLACE VIEW stg_order_lines AS
SELECT
    order_id,
    order_item_id,
    customer_id,
    order_ts,
    ship_ts,
    DATE_TRUNC('week',  order_ts)::DATE            AS order_week,   -- Monday start
    DATE_TRUNC('month', order_ts)::DATE            AS order_month,
    TRIM(product_name)                             AS product_name,
    category_name,
    department_name,
    customer_segment,
    market,
    order_region,
    order_country,
    shipping_mode,
    order_status,
    delivery_status,
    payment_type,
    days_scheduled,
    days_real,
    (days_real - days_scheduled)                   AS delay_days,
    late_delivery_risk                             AS is_late,
    CASE WHEN delivery_status = 'Shipping canceled' THEN 1 ELSE 0 END AS is_ship_canceled,
    quantity,
    sales::DECIMAL(12,2)                           AS sales,
    discount::DECIMAL(12,2)                        AS discount,
    discount_rate::DECIMAL(6,3)                    AS discount_rate,
    order_item_total::DECIMAL(12,2)                AS order_item_total,
    benefit_per_order::DECIMAL(12,2)               AS profit,
    CASE WHEN benefit_per_order < 0 THEN 1 ELSE 0 END AS is_loss
FROM supplychain_db.public.dataco_cleaned
WHERE order_id IS NOT NULL;

2. FACT AND DIMENSION TABLES

In [0]:
drop table if exists fact_order_lines cascade ;
create table fact_order_lines 
    DISTKEY(order_id)
    SORTKEY(order_ts)
AS 
SELECT * FROM stg_order_lines;

In [0]:
drop table dim_products;
create table dim_products
 DISTSTYLE ALL 
AS  
SELECT product_name,max(category_name) as category_name,max(department_name) as department_name 
 from fact_order_lines 
 group by product_name;

 select * from dim_products;

3.1 MART VIEWS: Reusable datasets for dashboard

In [0]:
-- Delviery Performance by month and shipping mode
create or replace view supplychain_db.public.mart_delivery_monthly as 
select 
    order_month,
    shipping_mode,
    count(*) as order_lines,
    sum(is_late) as late_lines,
    round(100.0 * sum(is_late)/count(*), 1) as late_pct,
    avg(cast(days_scheduled as decimal(6,2))) as avg_scheduled_days,
    avg(cast(days_real as decimal(6,2))) as avg_real_days
from supplychain_db.public.fact_order_lines
group by order_month, shipping_mode;

In [0]:
select * from mart_delivery_monthly limit 10

3.2 Regional demand and service

In [0]:
create or replace view mart_region_performance as 
select 
  market,
  order_region,
  count(distinct order_id) as orders,
  sum(sales) as sales,
  sum(quantity) as units,
  sum(profit) as profit,
  round(100.0 * sum(is_late)/count(*), 1) as late_pct,
  round(100.0 * sum(is_loss)/ count(*), 1) as loss_line_pct
from fact_order_lines
group by market,order_region;

In [0]:
select * from mart_region_performance limit 10;

3.3 product analysis

In [0]:
create or replace view product_pareto as 
with p as(

    select product_name,sum(sales) as sales, sum(profit) as profit, sum(quantity) as units
    from fact_order_lines 
    group by product_name
)

select product_name, sales,profit, units,
round(100.0 * sales/sum(sales) over(), 1) as sales_share_pct,
round(100.0 * sum(sales) over( order by sales desc rows unbounded preceding)/sum(sales) over(),1) as cumulative_share_pct 
from p;

3.4 Inventory health

In [0]:
select * from product_pareto limit 20;

3.4 Inventory health:

In [0]:
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT TRIM(product_name)) AS products,
       SUM(warehouse_inventory * cost_per_unit) AS total_cost
FROM raw_inventory_new;

In [0]:
-- DROP TABLE IF EXISTS raw_inventory_new cascade;
-- CREATE table raw_inventory_new (
--     product_name         VARCHAR(120),
--     year_month           INT,
--     warehouse_inventory  INT,
--     cost_per_unit        DECIMAL(10,4)
-- );

In [0]:
SELECT product_name, SUM(carrying_cost) AS cost
FROM mart_inventory_health GROUP BY 1 ORDER BY 2 DESC LIMIT 5;

In [0]:
SELECT column_name, data_type, numeric_scale
FROM svv_columns WHERE table_name = 'raw_inventory_new';

In [0]:
create or replace view supplychain_db.public.mart_inventory_health as 
with demand as (
select product_name, order_month,sum(quantity) as units_demanded from fact_order_lines group by 
product_name, order_month
),

inv as (
select trim(product_name) as product_name,to_date(year_month::varchar || '01', 'YYYYMMDD') as inv_month,
warehouse_inventory, cost_per_unit from raw_inventory_new

)
SELECT
    i.product_name,
    i.inv_month,
    i.warehouse_inventory,
    COALESCE(d.units_demanded, 0)                                    AS units_demanded,
    CASE WHEN i.warehouse_inventory = 0
          AND COALESCE(d.units_demanded, 0) > 0 THEN 1 ELSE 0 END    AS is_stockout,
    CASE WHEN i.warehouse_inventory > 0
          AND COALESCE(d.units_demanded, 0) = 0 THEN 1 ELSE 0 END    AS is_stock_no_demand,
    (i.warehouse_inventory * i.cost_per_unit)::DECIMAL(12,2)         AS carrying_cost
FROM inv i
LEFT JOIN demand d
       ON d.product_name = i.product_name
      AND d.order_month  = i.inv_month;

In [0]:
select *  from raw_inventory_new limit 10;

4. WBR / MBR VIEWS: week-over-week and month-over-month variance

In [0]:
CREATE OR REPLACE VIEW mart_wbr_weekly AS
WITH w AS (
    SELECT
        order_week,
        COUNT(DISTINCT order_id)                    AS orders,
        SUM(sales)                                  AS sales,
        SUM(profit)                                 AS profit,
        ROUND(100.0 * SUM(is_late) / COUNT(*), 1)   AS late_pct
    FROM fact_order_lines
    GROUP BY order_week
)
SELECT
    order_week,
    orders,
    LAG(orders) OVER (ORDER BY order_week)                          AS orders_prev_week,
    ROUND(100.0 * (orders - LAG(orders) OVER (ORDER BY order_week))
                / NULLIF(LAG(orders) OVER (ORDER BY order_week), 0), 1) AS orders_wow_pct,
    sales,
    profit,
    late_pct,
    late_pct - LAG(late_pct) OVER (ORDER BY order_week)             AS late_pct_wow_pts
FROM w;

In [0]:
CREATE OR REPLACE VIEW mart_mbr_monthly AS
WITH m AS (
    SELECT
        order_month,
        COUNT(DISTINCT order_id)                    AS orders,
        SUM(sales)                                  AS sales,
        SUM(profit)                                 AS profit,
        ROUND(100.0 * SUM(is_late) / COUNT(*), 1)   AS late_pct
    FROM fact_order_lines
    GROUP BY order_month
)
SELECT
    order_month,
    orders,
    ROUND(100.0 * (orders - LAG(orders) OVER (ORDER BY order_month))
                / NULLIF(LAG(orders) OVER (ORDER BY order_month), 0), 1) AS orders_mom_pct,
    sales,
    profit,
    late_pct,
    late_pct - LAG(late_pct) OVER (ORDER BY order_month)                 AS late_pct_mom_pts
FROM m;

In [0]:
select * from mart_mbr_monthly limit 10;

In [0]:
SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS orders, MIN(order_ts) AS first_order, MAX(order_ts) AS last_order
FROM fact_order_lines;

5.1 Promise vs actual: which shipping modes miss their schedule

In [0]:
SELECT
    shipping_mode,
    days_scheduled,
    days_real,
    COUNT(*)                                          AS order_lines,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY shipping_mode), 1) AS pct_of_mode
FROM fact_order_lines
GROUP BY shipping_mode, days_scheduled, days_real
ORDER BY shipping_mode, days_real

5.2What-if: reset the First Class promise from 1 day to 2 days

In [0]:
SELECT
    ROUND(100.0 * SUM(is_late) / COUNT(*), 1) AS current_late_pct,
    ROUND(100.0 * SUM(CASE WHEN shipping_mode = 'First Class'
                           THEN CASE WHEN days_real > 2 THEN 1 ELSE 0 END
                           ELSE is_late END) / COUNT(*), 1) AS late_pct_if_first_class_is_2_days
FROM fact_order_lines;

5.3 Stockout and overstock rates, plus carrying cost

In [0]:
SELECT
    SUM(is_stockout)                                              AS stockout_product_months,
    SUM(CASE WHEN units_demanded > 0 THEN 1 ELSE 0 END)           AS product_months_with_demand,
    ROUND(100.0 * SUM(is_stockout)
          / NULLIF(SUM(CASE WHEN units_demanded > 0 THEN 1 ELSE 0 END), 0), 1) AS stockout_pct,
    SUM(is_stock_no_demand)                                       AS stock_without_demand_months,
    SUM(carrying_cost)                                            AS total_carrying_cost
FROM mart_inventory_health;

5.4 Top products by carrying cost, with their share of demand

In [0]:
SELECT
    product_name,
    SUM(carrying_cost)  AS carrying_cost,
    SUM(units_demanded) AS units_demanded,
    SUM(is_stockout)    AS stockout_months
FROM mart_inventory_health
GROUP BY product_name
ORDER BY carrying_cost DESC
LIMIT 10;